# MLP Model Univariate

In this section we implement the MLP Model using the **TimeSeriesDataset** approach with one-hot encoding.

The MLP (Multi-Layer Perceptron) Forecaster is a feedforward neural network designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. The model flattens the entire input sequence into a single vector and processes it through fully connected layers, treating the sequence as a fixed-length feature vector without explicitly modeling temporal dependencies.

Key Insight: Each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific patterns alongside temporal and exogenous features.

**Layer Breakdown:**

- **Input Flattening**: Reshapes (batch_size, seq_length, input_size) → (batch_size, seq_length × input_size)
- **Hidden Layers**: 3 stacked fully connected layers with ReLU activation
- **Hidden Size**: 512 units per layer
- **Dropout**: Applied after each hidden layer (0.2)
- **Output Layer**: Single fully connected layer producing 1-step forecast

**Advantages:**

- **Simplicity**: Easy to implement, debug, and understand
- **Fast Training**: No recurrent computations, fully parallelizable
- **Global Pattern Recognition**: Sees entire sequence at once, captures long-range interactions
- **No Gradient Vanishing**: No backpropagation through time (unlike RNN/LSTM)

**Limitations:**

- **No Explicit Temporal Modeling**: Doesn't naturally capture sequential dependencies
- **High Parameter Count**: Flattening creates large first layer (scales with seq_length)
- **Poor for Long Sequences**: Parameter explosion and loss of temporal structure (best for <15 timesteps)
- **Fixed Input Length**: Requires same sequence length for all inputs

## Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `input_size` | - | Number of input features per timestep (Value + exogenous features + temporal features) |
| `seq_length` | 6 | Lookback window (number of historical timesteps) |
| `hidden_size` | 512 | Number of units in each hidden layer (controls model capacity) |
| `num_layers` | 3 | Number of hidden layers (depth of the network) |
| `dropout` | 0.2 | Dropout probability for regularization (prevents overfitting) |
| `batch_size` | 8 | Number of samples per training batch |
| `learning_rate` | 0.001 | Step size for optimizer (Adam) |
| `weight_decay` | 1e-5 | L2 regularization penalty on weights |

In [1]:
import torch
import torch.nn as nn

## Model

In [ ]:
class MLPForecaster(nn.Module):
    """
    MLP model for UNIVARIATE time series forecasting with one-hot encoding.
    Architecture: 
        Input Flattening -> MLP Layers (Linear -> ReLU -> Dropout) -> Output
    
    Takes input sequences with multiple features (Value + features + year + month + one-hot)
    and flattens them before processing through fully connected layers.
    
    Unlike MLPMultivariate:
    - Uses one-hot encoding to identify individual time series
    - Predicts one value at a time for a specific series
    - Processes entire sequence as flattened vector
    
    Good for:
    - Capturing non-linear patterns across the entire sequence
    - Faster training than RNN/LSTM for shorter sequences
    - Learning complex feature interactions
    """
    def __init__(self, input_size, seq_length, hidden_size=512, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            seq_length: Sequence length (lookback window)
            hidden_size: Number of units in each MLP layer
            num_layers: Number of MLP layers
            dropout: Dropout rate
        """
        super(MLPForecaster, self).__init__()
        
        self.input_size = input_size
        self.seq_length = seq_length
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Calculate input dimension after flattening
        self.input_dim = seq_length * input_size
        
        # Build MLP layers
        layers = []
        
        # First layer
        layers.append(nn.Linear(self.input_dim, hidden_size))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        
        self.mlp = nn.Sequential(*layers)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (batch_size, seq_length, input_size)
        
        Returns:
            predictions: (batch_size, 1) - single value prediction
        """
        batch_size = x.size(0)
        
        # Flatten entire sequence
        x_flat = x.reshape(batch_size, -1)  # (batch_size, seq_length * input_size)
        
        # Pass through MLP
        x = self.mlp(x_flat)  # (batch_size, hidden_size)
        
        # Output prediction
        out = self.fc(x)  # (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

In this section, the MLP model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well series-specific patterns are captured using only historical information and temporal context. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.


### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.59330 | 128 | 0.47862 | 256 | 0.00011 | 4 | 31.52s |
| 1 | 0.53338 | 32 | 0.23816 | 512 | 0.00138 | 2 | 30.09s |
| 2 | 0.60322 | 128 | 0.22516 | 128 | 0.00035 | 4 | 38.66s |



###Best Hyperparameters

Validation Loss: 0.53338

Parameters:
 - learning_rate: 0.00138
 - batch_size: 32
 - num_layers: 2
 - hidden_size: 512
 - dropout: 0.23816

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/mlp/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/mlp/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/mlp/fold3/fold_results.png)

### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 96336.24 | 310.38 | 136.16 | 0.8273 | 83.44% |
| Fold 2 | 87382.66 | 295.61 | 127.25 | 0.8257 | 77.51% |
| Fold 3 | 62290.40 | 249.58 | 103.11 | 0.8814 | 70.92% |
| **Average** | **82003.10 ± 17426.02** | **285.20 ± 31.17** | **122.18 ± 16.79** | **0.8448 ± 0.0315** | **77.29% ± 6.46%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 4.4% ± 1.2% | 59 |
| 10-20% | 11.7% ± 2.4% | 156 |
| 20-30% | 12.4% ± 1.9% | 166 |
| 30-40% | 10.7% ± 1.4% | 143 |
| >40% | 60.8% ± 5.8% | 811 |


**Comparison with Baseline:**

The MLP univariate model with one-hot encoding achieves an average SMAPE of **77.29% ± 6.46%**, which is **5.03 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). While this indicates that the baseline model outperforms the MLP on average, the difference is modest.


## Model Results with Exogenous Features

In this section, the MLP model is evaluated based on temporal features (value, year, and month), one-hot encoded series identifiers, and external economic indicators. This approach enables assessment of how well the model captures series-specific patterns by leveraging both historical information and exogenous economic features. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.51164 | 128 | 0.40536 | 256 | 0.00650 | 4 | 23.34s |
| 1 | 0.30996 | 128 | 0.20772 | 1024 | 0.00038 | 4 | 30.67s |
| 2 | 0.41197 | 128 | 0.37140 | 1024 | 0.00239 | 3 | 32.84s |




###Best Hyperparameters

Validation Loss: 0.30996

Parameters:
  - learning_rate: 0.00038
  - batch_size: 128
  - num_layers: 4
  - hidden_size: 1024
  - dropout: 0.20772

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/mlp_exo/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/mlp_exo/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/mlp_exo/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 125825.21 | 354.72 | 144.98 | 0.7744 | 81.81% |
| Fold 2 | 126342.88 | 355.45 | 147.74 | 0.7480 | 78.42% |
| Fold 3 | 72325.35 | 268.93 | 105.04 | 0.8623 | 75.62% |
| **Average** | **108164.48 ± 31071.59** | **326.37 ± 49.01** | **132.59 ± 22.57** | **0.7949 ± 0.0596** | **78.62% ± 3.11%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 2.5% ± 1.3% | 33 |
| 10-20% | 9.4% ± 4.0% | 126 |
| 20-30% | 12.2% ± 3.0% | 164 |
| 30-40% | 12.0% ± 0.6% | 160 |
| >40% | 63.9% ± 8.6% | 852 |


**Comparison with Baseline:**

The MLP univariate model with exogenous features achieves an average SMAPE of **78.62% ± 3.11%**, which is **6.36 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). While the inclusion of exogenous economic indicators improved validation loss during hyperparameter tuning, the model still underperforms the baseline on average. However, the exogenous features reduced SMAPE variance across folds, indicating more stable predictions across different time periods.